In [1]:
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
%matplotlib inline

import string
import os
import glob
from PIL import Image
from time import time
import collections
import random
import numpy as np
import json
import tensorflow as tf
import nltk
from nltk.translate.bleu_score import sentence_bleu
import requests

from keras import Input, layers
from keras import optimizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import LSTM, Embedding, Dense, Activation, Flatten, Reshape, Dropout
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import add
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from tqdm import tqdm

In [2]:
train_images = '/kaggle/input/coco-2017-dataset/coco2017/train2017'
test_images = '/kaggle/input/coco-2017-dataset/coco2017/test2017'
validation_images = '/kaggle/input/coco-2017-dataset/coco2017/val2017'
glove_path = '/kaggle/input/glove6b200d/glove.6B.200d.txt'
working_dir = '/kaggle/working/'

In [3]:
train_images_len = len(os.listdir(train_images))
test_images_len = len(os.listdir(test_images))
val_images_len = len(os.listdir(validation_images))
print(train_images_len)
print(test_images_len)
print(val_images_len)

118287
40670
5000


In [4]:
annotation_file = '/kaggle/input/coco-2017-dataset/coco2017/annotations/captions_train2017.json'

with open(annotation_file, 'r') as f:
    annotations = json.load(f)
print(annotations['annotations'][0])

{'image_id': 203564, 'id': 37, 'caption': 'A bicycle replica with a clock as the front wheel.'}


In [5]:
image_path_to_caption = collections.defaultdict(list)
for val in annotations['annotations']:
    caption = (f"{val['caption']}")
    image_path = train_images + '/'+'%012d.jpg' % (val['image_id'])  
    image_path_to_caption[image_path].append(caption)

In [6]:
annotations['annotations'][203564]


{'image_id': 241623,
 'id': 659699,
 'caption': 'A ceiling fan hangs over a clean kitchen.'}

In [7]:
train_images+'/%012d.jpg' % (241623) 


'/kaggle/input/coco-2017-dataset/coco2017/train2017/000000241623.jpg'

In [8]:
print(len(image_path_to_caption))
image_path_to_caption['/kaggle/input/coco-2017-dataset/coco2017/train2017/000000203564.jpg']

118287


['A bicycle replica with a clock as the front wheel.',
 'The bike has a clock as a tire.',
 'A black metal bicycle with a clock inside the front wheel.',
 'A bicycle figurine in which the front wheel is replaced with a clock\n',
 'A clock with the appearance of the wheel of a bicycle ']

In [9]:
image_path_to_caption = dict(image_path_to_caption)
print(type(image_path_to_caption))

<class 'dict'>


In [10]:
def id_caption(image_path_to_caption): 
    image_id_to_caption = collections.defaultdict(list)
    for (key,val) in image_path_to_caption.items(): 
        for values in val:
            x = key.split('/')[-1]
            x = x.split('.')[0]
            image_id_to_caption[x].append(values)

    # Mengonversi image_id_to_caption ke dict
    image_id_to_caption = dict(image_id_to_caption)
    return image_id_to_caption

In [11]:
table = str.maketrans('', '', string.punctuation)
for key, desc_list in image_path_to_caption.items():
    for i in range(len(desc_list)):
        desc = desc_list[i]
        desc = desc.split()
        desc = [word.lower() for word in desc]
        desc = [w.translate(table) for w in desc]
        desc_list[i] =  ' '.join(desc)

In [12]:
items = list(image_path_to_caption.items())
print(items[59142])

('/kaggle/input/coco-2017-dataset/coco2017/train2017/000000137711.jpg', ['a man standing on top of a bridge near a forest', 'tourists in rain gear at an observation post in a forest', 'a few people out in the rain in the woods', 'the lady in this photo was looking up at something', 'a group of people looking up at and around trees'])


In [13]:
vocabulary = set()
for key in image_path_to_caption.keys():
        [vocabulary.update(d.split()) for d in image_path_to_caption[key]]
print(len(vocabulary))

29075


In [14]:
image_id_to_caption = id_caption(image_path_to_caption)
print(len(image_id_to_caption))
print(type(image_id_to_caption))
print(list(image_id_to_caption.keys())[:5])

118287
<class 'dict'>
['000000203564', '000000322141', '000000016977', '000000106140', '000000571635']


In [15]:
lines = list()
for key, desc_list in image_id_to_caption.items():
    for desc in desc_list:
        lines.append(key + ' ' + desc)
new_descriptions = '\n'.join(lines)

print(type(new_descriptions))
print(new_descriptions[:400])

<class 'str'>
000000203564 a bicycle replica with a clock as the front wheel
000000203564 the bike has a clock as a tire
000000203564 a black metal bicycle with a clock inside the front wheel
000000203564 a bicycle figurine in which the front wheel is replaced with a clock
000000203564 a clock with the appearance of the wheel of a bicycle
000000322141 a room with blue walls and a white sink and door
00000032214


In [16]:
train = list(image_id_to_caption.keys())
print(train[0:5])

['000000203564', '000000322141', '000000016977', '000000106140', '000000571635']


In [17]:
train_img = list(image_path_to_caption.keys())
print(len(train_img))

118287


In [18]:
train_descriptions = dict()
for line in tqdm(new_descriptions.split('\n')):
    tokens = line.split()
    image_id, image_desc = tokens[0], tokens[1:]
    if image_id in train:
        if image_id not in train_descriptions:
            train_descriptions[image_id] = list()
        desc = 'startseq ' + ' '.join(image_desc) + ' endseq'
        train_descriptions[image_id].append(desc)

print(len(train_descriptions))

100%|██████████| 591753/591753 [14:36<00:00, 675.15it/s]

118287


In [19]:
list(train_descriptions.keys())[0:5]


['000000203564',
 '000000322141',
 '000000016977',
 '000000106140',
 '000000571635']

In [20]:
list(train_descriptions.values())[0:5]


[['startseq a bicycle replica with a clock as the front wheel endseq',
  'startseq the bike has a clock as a tire endseq',
  'startseq a black metal bicycle with a clock inside the front wheel endseq',
  'startseq a bicycle figurine in which the front wheel is replaced with a clock endseq',
  'startseq a clock with the appearance of the wheel of a bicycle endseq'],
 ['startseq a room with blue walls and a white sink and door endseq',
  'startseq blue and white color scheme in a small bathroom endseq',
  'startseq this is a blue and white bathroom with a wall sink and a lifesaver on the wall endseq',
  'startseq a blue boat themed bathroom with a life preserver on the wall endseq',
  'startseq a bathroom with walls that are painted baby blue endseq'],
 ['startseq a car that seems to be parked illegally behind a legally parked car endseq',
  'startseq two cars parked on the sidewalk on the street endseq',
  'startseq city street with parked cars and a bench endseq',
  'startseq cars try 

In [21]:
all_train_captions = []
for key, val in train_descriptions.items():
    for cap in val:
        all_train_captions.append(cap)

In [22]:
print(len(all_train_captions)) 
print(all_train_captions[:5])

591753
['startseq a bicycle replica with a clock as the front wheel endseq', 'startseq the bike has a clock as a tire endseq', 'startseq a black metal bicycle with a clock inside the front wheel endseq', 'startseq a bicycle figurine in which the front wheel is replaced with a clock endseq', 'startseq a clock with the appearance of the wheel of a bicycle endseq']


In [23]:
word_count_threshold = 10
word_counts = {}
nsents = 0
for sent in all_train_captions:
    nsents += 1
    for w in sent.split(' '):
        word_counts[w] = word_counts.get(w, 0) + 1
vocab = [w for w in word_counts if word_counts[w] >= word_count_threshold]

print('Vocabulary = %d' % (len(vocab)))

Vocabulary = 7442


In [24]:
ixtoword = {}
wordtoix = {}
ix = 1
for w in vocab:
    wordtoix[w] = ix
    ixtoword[ix] = w
    ix += 1

vocab_size = len(ixtoword) + 1

In [25]:
all_desc = list()
for key in train_descriptions.keys():
    [all_desc.append(d) for d in train_descriptions[key]]
lines = all_desc
max_length = max(len(d.split()) for d in lines)

print('Description Length: %d' % max_length)

Description Length: 51


In [26]:
embeddings_index = {} 
f = open(glove_path, encoding="utf-8")
for line in f:   
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs

In [27]:
embedding_dim = 200
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in wordtoix.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [28]:
embedding_matrix[5].shape

(200,)

# ResNet50 Model

In [29]:
from tensorflow.keras.applications import ResNet50

resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in resnet.layers:
    layer.trainable = False

image_input = resnet.input
image_features = resnet.output
image_features = tf.keras.layers.GlobalAveragePooling2D()(image_features)
image_features = Dense(256, activation='relu')(image_features)
encoder = Model(inputs=image_input, outputs=image_features)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [30]:
encoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 230, 230, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 112, 112, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_bn                  │ (None, 112, 112, 64)   │            256 │ conv1_conv[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_relu (Activation)   │ (None, 112, 112, 64)   │              0 │ conv1_bn[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 114, 114, 64)   │              0 │ conv1_relu[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 56, 56, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 56, 56, 64)     │          4,160 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 56, 56, 64)     │         36,928 │ conv2_block1_1_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv       │ (None, 56, 56, 256)    │         16,640 │ pool1_pool[0][0]       │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_3_conv       │ (None, 56, 56, 256)    │         16,640 │ conv2_block1_2_relu[0… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 24,112,256 (91.98 MB)

 Trainable params: 524,544 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [31]:
from keras.preprocessing.image import load_img, img_to_array
from keras.applications.resnet50 import preprocess_input
import numpy as np

def preprocess(image_path):
    # Load the image and resize it to (224, 224)
    img = load_img(image_path, target_size=(224, 224))  # Change target_size to (224, 224)
    
    # Convert the image to a numpy array
    x = img_to_array(img)
    
    # Expand dimensions to match the input shape (1, 224, 224, 3)
    x = np.expand_dims(x, axis=0)
    
    # Preprocess the image for ResNet50
    x = preprocess_input(x)
    
    return x

In [32]:
print(len(train_img))
print(train_img[0:5])

118287
['/kaggle/input/coco-2017-dataset/coco2017/train2017/000000203564.jpg', '/kaggle/input/coco-2017-dataset/coco2017/train2017/000000322141.jpg', '/kaggle/input/coco-2017-dataset/coco2017/train2017/000000016977.jpg', '/kaggle/input/coco-2017-dataset/coco2017/train2017/000000106140.jpg', '/kaggle/input/coco-2017-dataset/coco2017/train2017/000000571635.jpg']


In [33]:
def encode(image):
    # Preprocess the image
    image = preprocess(image)
    
    # Extract features using the encoder
    fea_vec = encoder.predict(image, verbose=0)
    
    # Reshape the feature vector
    fea_vec = np.reshape(fea_vec, fea_vec.shape[1])
    
    return fea_vec

In [34]:
encoding_train = {}
for img in tqdm(train_img):
    path = img.split('/')[-1]
    encoding_train[path] = encode(img)
features = encoding_train

100%|██████████| 118287/118287 [5:54:12<00:00,  5.57it/s]


In [35]:
import pickle 
with open('ResNet50model.pkl', 'wb') as file:
    pickle.dump(encoder, file)

In [36]:
import pickle 
with open('features.pkl', 'wb') as file:
    pickle.dump(features, file)